In [ ]:
from pyspark.sql import functions as F

In [ ]:
# start date and end date
start_date = '2024-01-01'
end_date = '2025-12-01'

In [ ]:
# generate one row per month start between start_date and end_date

df = (
    spark.sql(f"""
    SELECT
      explode(
          sequence(
              to_date('{start_date}', 'yyyy-MM-dd'),
              to_date('{end_date}', 'yyyy-MM-dd'),
              interval 1 month
          )
      ) as month_start_date
    """)
          )

In [ ]:
# add useful analytics columns
df = (
    df
    # surrogate key at month grain
    .withColumn('date_key', F.date_format('month_start_date', 'yyyyMMdd').cast('int'))
    .withColumn('year', F.year('month_start_date'))
    .withColumn('month', F.month('month_start_date'))
    .withColumn('month_name', F.date_format('month_start_date', 'MMMM'))
    .withColumn('month_short_name', F.date_format('month_start_date', 'MMM').cast('string'))
    .withColumn('quarter', F.quarter('month_start_date'))
    .withColumn('quarter_name', F.expr("case when quarter = 1 then 'Q1' when quarter = 2 then 'Q2' when quarter = 3 then 'Q3' when quarter = 4 then 'Q4' end"))
    .withColumn('year_quarter', F.concat(F.year('month_start_date'), F.lit('.'), F.expr("case when quarter = 1 then 'Q1' when quarter = 2 then 'Q2' when quarter = 3 then 'Q3' when quarter = 4 then 'Q4' end")))
)

In [ ]:
display(df)